# Input formats: `type_input`

`Makemodel` is the entry point for a model specification, and so far its input has always been
**Markdown**: prose, with the equations on lines starting with `>`. That is the main road through
ModelFlow, the one the `%%Makemymodel` magic takes, and the one the rest of this book uses.

The equations of a model do not always start life in a notebook, though. They may sit in a plain
ModelFlow text, in a file written by another program, or in another modelling system. The argument
`type_input` tells `Makemodel` what kind of text it is given, and four more arguments let you
describe a format of your own:

| Argument | Meaning |
|---|---|
| `type_input` | `'markdown'` (the default), `'modelflow'`, `'single'`, `'gekko'` or `'eviews'` |
| `eq_start` | String(s) which start an equation, e.g. `'FRML'`; text outside an equation is ignored |
| `eq_end` | String(s) which end an equation, e.g. `'$'` or `';'`; an equation can then span lines |
| `comment_start` | String(s) which start a comment running to the end of the line |
| `transform` | A function which translates each equation to ModelFlow form |

Whatever the input, the first step brings it to the same thing: **one ModelFlow statement per
line**. From there everything is shared -- `FRML` and `$` are added, lists, `do` loops and `doable`
are expanded, tagged equations are estimated, and the equations are normalized. So every feature of
the earlier chapters works whatever the input format.

In [1]:
import functools
import re

from modelconstruct_estimation import (Makemodel, extract_equations,
                                       gekko_to_modelflow, eviews_to_modelflow)

## Markdown -- the main road

In Markdown input the model **is** its documentation. Only lines starting with `>` are read as
model text; everything else is prose, headings, tables and formulas for the reader. A long
equation is continued on a line starting with `>>`.

In [2]:
Mmd = Makemodel("""
## Consumption
Households spend out of disposable income, and habits make last year's consumption matter:
><stoc> c = 0.6*yd
>>          + 0.3*c(-1)

## Income
Disposable income is income less taxes, and income is the sum of demand:
> yd = y - t
> y  = c + i + g
""")
Mmd.show

FRML <STOC> C = (0.6*YD+0.3*C(-1) + C_A)* (1-C_D)+ C_X*C_D  $
FRML <> YD = Y-T $
FRML <> Y = C+I+G $

FRML <CALC_ADD_FACTOR>  C_A = (C) - (0.6*YD+0.3*C(-1)) $


The same text renders as a document:

In [3]:
Mmd.render

## Consumption
Households spend out of disposable income, and habits make last year's consumption matter:

```text
><stoc> c = 0.6*yd
>>          + 0.3*c(-1)
```

## Income
Disposable income is income less taxes, and income is the sum of demand:

```text
> yd = y - t
> y  = c + i + g
```

Markdown input can also carry LaTeX: an `equation` environment with a `\label{eq:...}` is read as
an equation, with its tags in a `% @<...>` comment line, and list definitions can be written as
inline math. This is the format the `%%Makemymodel` magic passes on, and the rest of this chapter is
about the cases where the text is **not** Markdown.

## `type_input='modelflow'` -- plain ModelFlow text

Plain ModelFlow text has no prose, so there is no `>`. It can be written with or without the
`FRML <...> ... $` around each equation:

* **without**: one statement per line, the `FRML` and `$` are added;
* **with**: each statement ends with `$`, and may then run over several lines.

Use one or the other within a text: if there is a `$` anywhere, the text is split at `$`.

In [4]:
plain = """
c  = 0.6*yd + 0.3*c(-1)
yd = y - t
y  = c + i + g
"""

with_frml = """
FRML <> c  = 0.6*yd
           + 0.3*c(-1) $
FRML <> yd = y - t $
FRML <> y  = c + i + g $
"""

Mplain = Makemodel(plain, type_input='modelflow')
Mfrml = Makemodel(with_frml, type_input='modelflow')
print('the same model:', Mplain.normal_frml == Mfrml.normal_frml)
Mplain.show

the same model: True
FRML <> C = 0.6*YD+0.3*C(-1) $
FRML <> YD = Y-T $
FRML <> Y = C+I+G $


Lists can be passed separately with `list_defs=`, in the same plain form:

In [5]:
Mlist = Makemodel('doable <> profit__{banks} = revenue__{banks} - expenses__{banks}',
                  type_input='modelflow',
                  list_defs='list banks = banks : danske nordea')
Mlist.show

FRML <> PROFIT__DANSKE = REVENUE__DANSKE-EXPENSES__DANSKE $
FRML <> PROFIT__NORDEA = REVENUE__NORDEA-EXPENSES__NORDEA $


`type_input='single'` is for text which is already a complete stream of `FRML ... $` statements:
the line breaks are simply removed.

## Your own format: separators

When equations come from somewhere else, describe where they are:

* `eq_end` -- the string which ends an equation. Without it, each line is one equation.
* `eq_start` -- the string which starts an equation. With it, all text **between** equations is
  ignored -- titles, notes, output from the program which wrote the file.
* `comment_start` -- the string which starts a comment; the rest of the line is dropped.

Each can be one string or a list of strings. Markers are matched regardless of case, and a marker
made of letters only matches as a whole word: the start marker `EQ` does not match inside the
variable `EQUITY`.

As an example, a model listing written by some other program. Each equation starts with
`EQUATION` and ends with `END`, and there is text in between which is not part of the model:

In [6]:
listing = """
Model listing, written 2026-10-01
EQUATION c = 0.6*yd + 0.3*c(-1) END
  (estimated 1990-2019)
EQUATION yd = y - t END
EQUATION y = c + i + g
         END
3 equations written
"""

Mlisting = Makemodel(listing, eq_start='EQUATION', eq_end='END')
Mlisting.show

FRML <> C = 0.6*YD+0.3*C(-1) $
FRML <> YD = Y-T $
FRML <> Y = C+I+G $


Giving any of the four extraction arguments switches extraction on, so `type_input` can be left
at its default. To see what the extraction step itself makes of a text, call
`extract_equations` -- the function `Makemodel` uses:

In [7]:
print(extract_equations(listing, start='EQUATION', end='END'))

c = 0.6*yd + 0.3*c(-1)
yd = y - t
y = c + i + g


## Your own format: a transformation function

The separators find the equations; a **transformation function** translates each one. It is
called with one equation at a time -- the markers removed, comments dropped and all whitespace
collapsed to single spaces -- and returns it in ModelFlow form, `lhs = rhs` or
`<tags> lhs = rhs`. If it returns `None` or an empty string, the statement is dropped.

Here equations end with `;`, comments start with `#`, lags are written in square brackets, and a
keyword at the start says what each statement is: `BEH` for a behavioural equation, `IDN` for an
identity. There is also a `PRINT` statement, which is not an equation at all.

In [8]:
source = """
# A small model in some other format
BEH c  = 0.6*yd + 0.3*c[-1] ;       # consumption
BEH i  = 0.2*(y[-1] - y[-2]) ;      # investment
IDN yd = y - t ;
IDN y  = c + i + g ;
PRINT y ;
"""

print(extract_equations(source, end=';', comment='#'))

BEH c = 0.6*yd + 0.3*c[-1]
BEH i = 0.2*(y[-1] - y[-2])
IDN yd = y - t
IDN y = c + i + g
PRINT y


That is what the transformation function receives, one line at a time. It translates the lags,
turns the keyword into ModelFlow tags -- `<stoc>` gives a behavioural equation an add factor and
an exogenizing dummy -- and drops everything else:

In [9]:
def my_transform(stmt):
    """One equation of the BEH/IDN format -> ModelFlow"""
    keyword, body = stmt.split(maxsplit=1)
    body = re.sub(r'\[\s*([+-]?\d+)\s*\]', r'(\1)', body)    # c[-1] -> c(-1)
    if keyword.upper() == 'BEH':
        return f'<stoc> {body}'
    if keyword.upper() == 'IDN':
        return body
    return None                                              # PRINT and the like


Mown = Makemodel(source, eq_end=';', comment_start='#', transform=my_transform)
Mown.show

FRML <STOC> C = (0.6*YD+0.3*C(-1) + C_A)* (1-C_D)+ C_X*C_D  $
FRML <STOC> I = (0.2*(Y(-1)-Y(-2)) + I_A)* (1-I_D)+ I_X*I_D  $
FRML <> YD = Y-T $
FRML <> Y = C+I+G $

FRML <CALC_ADD_FACTOR>  C_A = (C) - (0.6*YD+0.3*C(-1)) $ 
FRML <CALC_ADD_FACTOR>  I_A = (I) - (0.2*(Y(-1)-Y(-2))) $


The function can return any tag, also `<est=...>`, so equations from another system can be
estimated while the model is built, exactly as in the *Estimation* chapter.

## Presets: Gekko and EViews

Two formats come ready-made. `type_input='gekko'` and `type_input='eviews'` fill in the separators
and the transformation function:

| `type_input` | `eq_start` | `eq_end` | `comment_start` | `transform` |
|---|---|---|---|---|
| `'gekko'` | `'FRML'` | `'$'` | `'()'` | `gekko_to_modelflow` |
| `'eviews'` | -- | -- (one per line) | `"'"` | `eviews_to_modelflow` |

### Gekko

A Gekko `FRML` carries a code, like `_SJRD`. `gekko_to_modelflow` reads the letters after the
first one: `J` gives an add factor, `D` an exogenizing dummy, so `J` and `D` together become
`<stoc>`, `J` alone `<add>`, `D` alone `<exo>`. It also translates `x[-1]` to `x(-1)`, `^` to
`**`, `dif(` to `diff(` and `pch(` to `pct_growth(`.

In [10]:
gekko_text = """
() A small model in Gekko format
FRML _SJRD log(c) = 0.5*log(yd) + 0.4*log(c[-1]) + 0.1 $
FRML _I    yd = y - t $
FRML _GJR  y = c + i + g $
"""

print(extract_equations(gekko_text, start='FRML', end='$', comment='()',
                        transform=gekko_to_modelflow))

<STOC> LOG(C) = 0.5*LOG(YD) + 0.4*LOG(C(-1)) + 0.1
<> YD = Y - T
<ADD> Y = C + I + G


In [11]:
Mgekko = Makemodel(gekko_text, type_input='gekko')
Mgekko.show

FRML <STOC> C = (EXP(C_A+ (0.5*LOG(YD)+0.4*LOG(C(-1))+0.1) )) * (1-C_D)+ C_X*C_D  $
FRML <> YD = Y-T $
FRML <ADD> Y = (C+I+G + Y_A)                                $

FRML <CALC_ADD_FACTOR>  C_A = - ((0.5*LOG(YD)+0.4*LOG(C(-1))+0.1)) +LOG(C) $ 
FRML <CALC_ADD_FACTOR>  Y_A = (Y) - (C+I+G) $


### EViews

:::{warning}
The EViews input is still work in progress: not all EViews functions and equation forms are
translated yet. The cells below therefore catch errors and print them instead of stopping the
notebook.
:::


EViews equations come one per line, with `'` starting a comment. `eviews_to_modelflow` gives an
`@identity` equation no tags, translates the EViews functions (`d(` to `diff(`, `@pc` to
`pct_growth`, `^` to `**`, ...), and gives every other equation the tags in `behavioral_opts`,
by default none:

In [12]:
eviews_text = """
' A small model in EViews format
log(c) = 0.5*log(yd) + 0.4*log(c(-1)) + 0.1
@identity yd = y - t
@identity y = c + i + g
"""

try:   # EViews input is work in progress
    Meviews = Makemodel(eviews_text, type_input='eviews')
    Meviews.show
except Exception as e:
    print(f'EViews input failed: {type(e).__name__}: {e}')

FRML <> C = EXP( (0.5*LOG(YD)+0.4*LOG(C(-1))+0.1) ) $
FRML <> YD = Y-T $
FRML <> Y = C+I+G $


### Adjusting a preset

A preset only fills in what is not given, so any of the four arguments can be overridden while the
rest of the preset is kept. The translation functions take options of their own, which
`functools.partial` binds:

* `gekko_to_modelflow(stmt, code_map=...)` -- a dict from code to tags, or a function of the code;
  codes missing from a dict get the default translation;
* `eviews_to_modelflow(stmt, behavioral_opts=...)` -- the tags for the behavioural equations.

In [13]:
try:   # EViews input is work in progress
    Meviews_stoc = Makemodel(eviews_text, type_input='eviews',
                             transform=functools.partial(eviews_to_modelflow, behavioral_opts='<stoc>'))
    Meviews_stoc.show
except Exception as e:
    print(f'EViews input failed: {type(e).__name__}: {e}')

FRML <STOC> C = (EXP(C_A+ (0.5*LOG(YD)+0.4*LOG(C(-1))+0.1) )) * (1-C_D)+ C_X*C_D  $
FRML <> YD = Y-T $
FRML <> Y = C+I+G $

FRML <CALC_ADD_FACTOR>  C_A = - ((0.5*LOG(YD)+0.4*LOG(C(-1))+0.1)) +LOG(C) $

In [14]:
Mgekko_exo = Makemodel(gekko_text, type_input='gekko',
                       transform=functools.partial(gekko_to_modelflow, code_map={'_GJR': '<exo>'}))
Mgekko_exo.show

FRML <STOC> C = (EXP(C_A+ (0.5*LOG(YD)+0.4*LOG(C(-1))+0.1) )) * (1-C_D)+ C_X*C_D  $
FRML <> YD = Y-T $
FRML <EXO> Y = (C+I+G)* (1-Y_D)+ Y_X*Y_D $

FRML <CALC_ADD_FACTOR>  C_A = - ((0.5*LOG(YD)+0.4*LOG(C(-1))+0.1)) +LOG(C) $


A transformation function of your own can also build on a preset one -- do your own translations
first, then hand the equation on:

In [15]:
def my_gekko(stmt):
    """Gekko, with the variable name YD written as YDISP in the source"""
    return gekko_to_modelflow(re.sub(r'\bydisp\b', 'yd', stmt, flags=re.IGNORECASE))


Makemodel(gekko_text.replace('yd', 'ydisp'), type_input='gekko', transform=my_gekko).show

FRML <STOC> C = (EXP(C_A+ (0.5*LOG(YD)+0.4*LOG(C(-1))+0.1) )) * (1-C_D)+ C_X*C_D  $
FRML <> YD = Y-T $
FRML <ADD> Y = (C+I+G + Y_A)                                $

FRML <CALC_ADD_FACTOR>  C_A = - ((0.5*LOG(YD)+0.4*LOG(C(-1))+0.1)) +LOG(C) $ 
FRML <CALC_ADD_FACTOR>  Y_A = (Y) - (C+I+G) $

## Quick reference

```python
Makemodel(text)                                   # Markdown: equations on lines starting with >
Makemodel(text, type_input='modelflow')           # plain ModelFlow, with or without FRML ... $
Makemodel(text, type_input='gekko')               # Gekko FRML ... $ with codes
Makemodel(text, type_input='eviews')              # EViews, one equation per line
Makemodel(text, eq_start='EQUATION', eq_end='END',
          comment_start='#', transform=my_transform)   # a format of your own
```

| What you need | How |
|---|---|
| Equations spanning several lines | `eq_end=` |
| Ignore text between equations | `eq_start=` |
| Comments | `comment_start=` |
| Several markers | a list, e.g. `eq_end=['$', ';']` |
| Translate or tag each equation | `transform=`, a function `str -> str` |
| Drop a statement | let `transform` return `None` |
| Change a preset | pass the argument, the rest of the preset is kept |
| Options of a preset function | `functools.partial(gekko_to_modelflow, code_map=...)` |
| Lists | `list_defs=`, Markdown for Markdown input, plain text otherwise |
| See what was extracted | `extract_equations(text, start=..., end=..., comment=..., transform=...)` |